# 05 · Video pipeline design / Diseño de un pipeline de vídeo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb)

*Part III · group · 15 min*

A video pipeline is the path from a **video file** to the **tensor a model actually receives**.

The key idea is simple:

> **Every pipeline decision chooses what information is kept, transformed, or discarded.**

> 🇪🇸 Un pipeline de video es el camino desde un **archivo de video** hasta el **tensor que realmente recibe un modelo**.
>
> La idea central es sencilla:
>
> **Cada decisión del pipeline determina qué información se conserva, se transforma o se descarta.**

## What you will be able to do / Lo que podrás hacer

- Follow one verified real video from file bytes to a tensor.
- Read `(T, H, W, C)` as a sentence and explain every axis.
- Measure how many recorded frames are kept when the video is sampled.
- Explain why one model may **collapse time** while another must **preserve time**.
- Compare variable-length padding with fixed-frame sampling.
- Reason about an additional synchronized camera axis.

> 🇪🇸
>
> - Seguir un video real verificado desde los bytes del archivo hasta un tensor.
> - Leer `(T,H,W,C)` como una frase y explicar cada eje.
> - Medir cuántos fotogramas grabados se conservan al muestrear un video.
> - Explicar por qué un modelo puede **colapsar el tiempo** mientras otro debe **conservarlo**.
> - Comparar padding para longitudes variables con muestreo de un número fijo de fotogramas.
> - Razonar sobre un eje adicional de cámaras sincronizadas.

## The video axes / Los ejes del video

A decoded colour video is commonly represented as:

`(T, H, W, C)`

| Axis / Eje | English | Español | Simple question / Pregunta sencilla |
|---|---|---|---|
| `T` | time / frames | tiempo / fotogramas | Which recorded moment? / ¿Qué momento grabado? |
| `H` | height | alto | Which pixel row? / ¿Qué fila de píxeles? |
| `W` | width | ancho | Which pixel column? / ¿Qué columna de píxeles? |
| `C` | colour channels | canales de color | Red, green, or blue? / ¿Rojo, verde o azul? |

Read:

`(16, 540, 960, 3)`

as:

**16 retained moments × 540 pixel rows × 960 pixel columns × 3 colour channels**

> 🇪🇸 Lee `(16,540,960,3)` como:
>
> **16 momentos conservados × 540 filas de píxeles × 960 columnas de píxeles × 3 canales de color**

## Setup / Preparación

Run this cell first.

It downloads one **real CC0 WebM video from Wikimedia Commons**, verifies its SHA-256 checksum, decodes the full stream to count the recorded frames, and keeps only a sparse set of real frames in memory.

The full 720-frame video is deliberately **not** materialized as one large `(T,H,W,C)` NumPy tensor. Avoiding that large allocation is already a pipeline-design choice.

> 🇪🇸 Ejecuta primero esta celda.
>
> Descarga un **video WebM real CC0 de Wikimedia Commons**, verifica su checksum SHA-256, decodifica el flujo completo para contar los fotogramas grabados y conserva en memoria solo una muestra dispersa de fotogramas reales.
>
> El video completo de 720 fotogramas no se materializa deliberadamente como un gran tensor NumPy `(T,H,W,C)`. Evitar esa asignación de memoria ya es una decisión de diseño del pipeline.

In [ ]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Real clip: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0.
# https://commons.wikimedia.org/wiki/File:Tormenta_en_l%27Almadrava.webm
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)

VIDEO_SHA256 = (
    "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
)

UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"

def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    # Verify the real file, decode the full stream, retain sparse real frames.
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    kept_frames = []
    kept_source_indices = []
    total_frames = 0

    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        total_frames = i + 1

        if i % stride == 0 and len(kept_frames) < n_frames:
            kept_frames.append(frame)
            kept_source_indices.append(i)

    clip = np.stack(kept_frames)

    return clip, np.asarray(kept_source_indices), total_frames

clip, kept_source_indices, total_frames = fetch_verified_video(
    VIDEO_URL,
    VIDEO_SHA256,
)

assert clip.shape == (16, 540, 960, 3), clip.shape
assert total_frames == 720, total_frames

print("Retained tensor / Tensor conservado:", clip.shape, clip.dtype)
print("Recorded source frames / Fotogramas grabados:", total_frames)
print(
    "Retained source indices / Índices originales conservados:",
    kept_source_indices.tolist(),
)
print(
    "RAM retained / RAM conservada:",
    f"{clip.nbytes / 1024**2:.1f} MB",
)
print()
print("EN: Setup ready with verified real video data.")
print("ES: Preparación lista con datos reales de video verificados.")

## Why this matters / Por qué esto importa

The camera recorded **720 frames**.

This notebook keeps **16 real frames** in memory:

`(16, 540, 960, 3)`

So the tensor is convenient and much smaller, but it does **not** contain every recorded timestep.

This is the central idea:

> **Efficiency has a cost. Always ask what information was removed to make the tensor smaller or simpler.**

### Predict → Run → Explain / Predice → Ejecuta → Explica

Before every exercise:

1. predict the shape;
2. name every axis;
3. run the code;
4. explain what was kept;
5. explain what was lost or transformed.

> 🇪🇸 La cámara grabó **720 fotogramas**, pero este cuaderno conserva **16 fotogramas reales** en memoria.
>
> El tensor es más pequeño y manejable, pero ya no contiene todos los instantes grabados.
>
> **La eficiencia tiene un costo. Pregunta siempre qué información se eliminó para hacer el tensor más pequeño o más sencillo.**

## Exercise 1 — watch a real video become a tensor / Ejercicio 1 — observa cómo un video real se convierte en tensor

`clip` contains 16 real frames sampled from the verified 720-frame source video.

### Predict first / Predice primero

For:

`clip.shape = (16, 540, 960, 3)`

answer:

1. What does each axis count?
2. What percentage of the 720 recorded frames is present in `clip`?
3. Is `clip[1]` the original source frame `1`?

> 🇪🇸 `clip` contiene 16 fotogramas reales muestreados del video verificado de 720 fotogramas.
>
> Antes de ejecutar, responde:
>
> 1. ¿Qué cuenta cada eje?
> 2. ¿Qué porcentaje de los 720 fotogramas grabados está presente en `clip`?
> 3. ¿`clip[1]` corresponde al fotograma original `1`?

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# Print clip.shape and clip.dtype.
# Name axes 0, 1, 2, and 3.
#
# ES:
# Imprime clip.shape y clip.dtype.
# Nombra los ejes 0, 1, 2 y 3.
#
# TODO 2 / TAREA 2
#
# EN:
# Using len(clip) and total_frames, compute:
# - fraction of recorded frames retained
# - fraction of recorded frames not retained
#
# ES:
# Usando len(clip) y total_frames, calcula:
# - fracción de fotogramas grabados conservados
# - fracción de fotogramas grabados no conservados
#
# TODO 3 / TAREA 3
#
# EN:
# Inspect kept_source_indices.
# Explain why clip[1] is NOT source frame 1.
#
# ES:
# Inspecciona kept_source_indices.
# Explica por qué clip[1] NO es el fotograma original 1.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

retained_fraction = len(clip) / total_frames
not_retained_fraction = 1 - retained_fraction

print("Shape / Forma:", clip.shape)
print("dtype:", clip.dtype)
print("Axes / Ejes: (T, H, W, C)")
print("EN: retained time × height × width × colour.")
print("ES: tiempo conservado × alto × ancho × color.")
print()

print(
    "Retained recorded frames / Fotogramas grabados conservados:",
    f"{retained_fraction:.2%}",
)
print(
    "Not retained / No conservados:",
    f"{not_retained_fraction:.2%}",
)
print()

print(
    "clip[1] came from source frame / clip[1] proviene del fotograma original:",
    int(kept_source_indices[1]),
)

print()
print("EN: clip index and source-frame index are different coordinate systems.")
print("ES: el índice dentro de clip y el índice del video original son sistemas de coordenadas diferentes.")

fig, axes = plt.subplots(2, 1, figsize=(11, 6))

axes[0].scatter(
    np.arange(total_frames),
    np.zeros(total_frames),
    s=7,
    alpha=0.18,
    label="recorded / grabado",
)

axes[0].scatter(
    kept_source_indices,
    np.zeros_like(kept_source_indices),
    s=45,
    label="retained / conservado",
)

axes[0].set_yticks([])
axes[0].set_xlim(-5, total_frames + 5)
axes[0].set_xlabel("Source frame index / Índice del fotograma original")
axes[0].set_title(
    f"Sampling / Muestreo: {len(clip)} of/de {total_frames} "
    f"({retained_fraction:.1%})"
)
axes[0].legend(loc="upper right")

preview_slots = [0, 5, 10, 15]
strip = np.concatenate([clip[k] for k in preview_slots], axis=1)

axes[1].imshow(strip)
axes[1].set_title(
    "Four retained real frames / Cuatro fotogramas reales conservados — "
    + ", ".join(
        str(int(kept_source_indices[k])) for k in preview_slots
    )
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Interactive retained-frame browser / Explorador interactivo de fotogramas conservados

Move the slider — or press **Play** — to browse the 16 retained real frames.

Watch two numbers carefully:

- `clip[k]` = position inside the small tensor;
- `source frame` = position in the original 720-frame video.

For example, `clip[1]` comes from source frame `45`, not source frame `1`.

> 🇪🇸 Mueve el control — o presiona **Play** — para recorrer los 16 fotogramas reales conservados.
>
> Observa dos números:
>
> - `clip[k]` = posición dentro del tensor pequeño;
> - `source frame` = posición dentro del video original de 720 fotogramas.

In [ ]:
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(clip) - 1,
    step=1,
    description="clip[k]:",
    continuous_update=False,
    style={"description_width": "80px"},
)

frame_play = widgets.Play(
    value=0,
    min=0,
    max=len(clip) - 1,
    step=1,
    interval=500,
    description="Play",
)

widgets.jslink(
    (frame_play, "value"),
    (frame_slider, "value"),
)

def show_retained_frame(k):
    plt.close("all")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(clip[k])
    ax.set_title(
        f"clip[{k}] → source frame / fotograma original "
        f"{int(kept_source_indices[k])} of/de {total_frames}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("Tensor position / Posición en tensor:", k)
    print("Source position / Posición original:", int(kept_source_indices[k]))
    print("Frame shape / Forma del fotograma:", clip[k].shape)

frame_output = widgets.interactive_output(
    show_retained_frame,
    {"k": frame_slider},
)

display(
    widgets.VBox([
        widgets.HBox([frame_play, frame_slider]),
        frame_output,
    ])
)

### Sampling calculator / Calculadora de muestreo

The real tensor above keeps 16 frames.

Use the slider below to ask a design question:

**“If I kept a different number of positions from a 720-frame video, what fraction of recorded timesteps would that represent?”**

This calculator changes only the **sampling plan**. It does not invent or decode additional frames.

> 🇪🇸 El tensor real anterior conserva 16 fotogramas.
>
> Usa el control para responder:
>
> **“Si conservara un número diferente de posiciones de un video de 720 fotogramas, ¿qué fracción de los instantes grabados representaría?”**
>
> Esta calculadora solo cambia el **plan de muestreo**; no inventa ni decodifica fotogramas adicionales.

In [ ]:
keep_slider = widgets.IntSlider(
    value=16,
    min=1,
    max=120,
    step=1,
    description="Keep / Conservar:",
    continuous_update=False,
    style={"description_width": "120px"},
)

def sampling_calculator(n_keep):
    fraction = n_keep / total_frames
    not_kept = 1 - fraction

    # Evenly spaced design positions for visualization only.
    planned = np.linspace(
        0,
        total_frames - 1,
        n_keep,
        dtype=int,
    )

    plt.close("all")
    fig, ax = plt.subplots(figsize=(10, 1.8))

    ax.scatter(
        np.arange(total_frames),
        np.zeros(total_frames),
        s=5,
        alpha=0.12,
    )

    ax.scatter(
        planned,
        np.zeros_like(planned),
        s=28,
    )

    ax.set_yticks([])
    ax.set_xlim(-5, total_frames + 5)
    ax.set_xlabel("Recorded frame index / Índice de fotograma grabado")
    ax.set_title(
        f"Design plan / Plan de diseño: {n_keep} of/de {total_frames}"
    )

    plt.tight_layout()
    plt.show()

    print(f"Retained fraction / Fracción conservada: {fraction:.2%}")
    print(f"Not retained / No conservada: {not_kept:.2%}")
    print("EN: fewer retained positions reduce memory but observe time more sparsely.")
    print("ES: menos posiciones conservadas reducen memoria, pero observan el tiempo de forma más dispersa.")

sampling_output = widgets.interactive_output(
    sampling_calculator,
    {"n_keep": keep_slider},
)

display(widgets.VBox([keep_slider, sampling_output]))

<details>
<summary><strong>What did Exercise 1 show? / ¿Qué mostró el Ejercicio 1?</strong></summary>

`clip` is not a smaller copy containing the first 16 frames.

It is a **sampled sequence**:

`clip[0] → source frame 0`

`clip[1] → source frame 45`

`clip[2] → source frame 90`

and so on.

The tensor therefore contains real measured images, but many recorded timesteps are absent.

> 🇪🇸 `clip` no es una copia pequeña que contenga los primeros 16 fotogramas.
>
> Es una **secuencia muestreada**:
>
> `clip[0] → fotograma original 0`
>
> `clip[1] → fotograma original 45`
>
> `clip[2] → fotograma original 90`
>
> El tensor contiene imágenes reales medidas, pero muchos instantes grabados no están presentes.

</details>

## Exercise 2 — one real input, two different goals / Ejercicio 2 — una entrada real, dos objetivos diferentes

Both systems can start from decoded video axes:

`(T, H, W, C)`

But they answer different questions.

### System A — short-video recommender / Recomendador de videos cortos

Question:

**“What is this whole video about?”**

A useful output may be **one embedding per video**:

`(N, embedding)`

Time is summarized.

### System B — surgical phase labelling / Etiquetado de fases quirúrgicas

Question:

**“What phase is happening at each timestep?”**

A useful output keeps time:

`(N, T, classes)`

Time must survive.

> 🇪🇸 Ambos sistemas pueden comenzar con `(T,H,W,C)`, pero responden preguntas diferentes.
>
> Un recomendador puede necesitar **una representación para todo el video**.
>
> Un sistema de fases quirúrgicas necesita una predicción **en cada instante**, por lo que el eje temporal debe conservarse.
>
> Estos son **escenarios de diseño**, no nuevos conjuntos de datos.

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# Fill in one defensible shape at each stage for BOTH systems.
# Next to every shape, write what each axis means.
#
# --- Tech: short-video recommender -------------------------------------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...
#
# --- Biotech: surgical phase labelling --------------------------------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...
#
# Then answer:
# Which system intentionally removes the time axis at the output?
#
# ES:
# Completa una forma coherente en cada etapa para AMBOS sistemas.
# Junto a cada forma, escribe qué significa cada eje.
#
# Después responde:
# ¿Qué sistema elimina intencionalmente el eje temporal en su salida?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

# One defensible design. Other sizes can also be correct when the
# axis meanings and system goals remain internally consistent.

# --- Tech: short-video recommender -------------------------------------------
# raw file          : bytes on disk / bytes en disco
# decoded frames    : (T, H, W, C)
# preprocessed batch: (32, 8, 224, 224, 3)  -> N, T, H, W, C
# model input       : (32, 8, 224, 224, 3)
# model output      : (32, 512)              -> N, embedding
# Time is summarized into one representation per video.

# --- Biotech: surgical phase labelling ---------------------------------------
# raw file          : bytes on disk / bytes en disco
# decoded frames    : (T, H, W, C)
# preprocessed batch: (4, 64, 224, 224, 3)  -> N, T, H, W, C
# model input       : (4, 64, 224, 224, 3)
# model output      : (4, 64, 12)            -> N, T, classes
# Time remains because the predicted phase may change at each timestep.

print("Real anchor / Ancla real:", clip.shape, "-> (T, H, W, C)")
print()
print("Tech output / Salida tech:", (32, 512))
print("EN: one vector per video; time has been summarized.")
print("ES: un vector por video; el tiempo ha sido resumido.")
print()
print("Biotech output / Salida biotech:", (4, 64, 12))
print("EN: one class distribution per timestep; time remains.")
print("ES: una distribución de clases por instante; el tiempo permanece.")

### Interactive “does time survive?” explorer / Explorador interactivo “¿sobrevive el tiempo?”

Switch between the two systems.

The important question is not whether one shape is “better.” The question is:

**Does the task require an answer for the whole video or for every timestep?**

> 🇪🇸 Cambia entre los dos sistemas.
>
> La pregunta importante no es cuál forma es “mejor”, sino:
>
> **¿La tarea necesita una respuesta para todo el video o una respuesta para cada instante?**

In [ ]:
system_toggle = widgets.ToggleButtons(
    options=[
        ("Recommender / Recomendador", "recommender"),
        ("Surgical phases / Fases quirúrgicas", "surgical"),
    ],
    value="recommender",
    description="System / Sistema:",
    style={"description_width": "115px"},
)

def explain_system(system):
    if system == "recommender":
        stages = [
            ("Decoded / Decodificado", "(T,H,W,C)"),
            ("Batch", "(N,T,H,W,C)"),
            ("Output / Salida", "(N,embedding)"),
        ]
        en = "Time is summarized because the output describes the whole video."
        es = "El tiempo se resume porque la salida describe el video completo."
        survives = "NO"
    else:
        stages = [
            ("Decoded / Decodificado", "(T,H,W,C)"),
            ("Batch", "(N,T,H,W,C)"),
            ("Output / Salida", "(N,T,classes)"),
        ]
        en = "Time survives because the output can change at every timestep."
        es = "El tiempo permanece porque la salida puede cambiar en cada instante."
        survives = "YES / SÍ"

    print("Pipeline / Pipeline")
    for label, shape in stages:
        print(f"{label:22s} → {shape}")

    print()
    print("Time survives? / ¿Sobrevive el tiempo?:", survives)
    print("EN:", en)
    print("ES:", es)

system_output = widgets.interactive_output(
    explain_system,
    {"system": system_toggle},
)

display(widgets.VBox([system_toggle, system_output]))

<details>
<summary><strong>Why can both designs be correct? / ¿Por qué ambos diseños pueden ser correctos?</strong></summary>

A tensor shape is part of the **task definition**.

For whole-video classification or retrieval, the model may deliberately summarize time.

For timestep-level labelling, removing time would destroy the coordinate needed to attach a prediction to each moment.

> 🇪🇸 La forma del tensor forma parte de la **definición de la tarea**.
>
> Para clasificación o recuperación de videos completos, un modelo puede resumir deliberadamente el tiempo.
>
> Para etiquetado por instante, eliminar el tiempo destruiría la coordenada necesaria para asociar una predicción con cada momento.

</details>

## Exercise 3 — variable-length clips: pad or sample? / Ejercicio 3 — clips de longitud variable: ¿padding o muestreo?

Real systems receive videos with different durations.

To make the memory trade-off visible **without allocating a gigantic image tensor**, we use a stress-test design scenario:

- 30 seconds,
- 45 seconds,
- 2 minutes,
- 4 hours,

all at 30 frames per second.

These durations are **chosen design inputs**, not measurements from the Wikimedia video.

### Option A — padding

Make every sequence as long as the longest clip and use a mask to mark which positions are real.

### Option B — fixed sampling

Keep exactly 64 positions from every clip.

This gives predictable memory, but long clips are observed much more sparsely.

> 🇪🇸 Los sistemas reales reciben videos con duraciones diferentes.
>
> Para visualizar el costo sin crear un tensor de imágenes gigantesco, usamos un escenario de estrés de 30 s, 45 s, 2 min y 4 h a 30 fps.
>
> Estas duraciones son **entradas de diseño elegidas**, no mediciones del video de Wikimedia.
>
> **Padding:** lleva todas las secuencias hasta la longitud máxima y usa una máscara.
>
> **Muestreo fijo:** conserva exactamente 64 posiciones de cada clip; la memoria es predecible, pero los videos largos se observan de forma mucho más dispersa.

In [ ]:
# TODO 5 / TAREA 5
#
# EN:
# Convert [30 s, 45 s, 2 min, 4 h] at 30 fps into frame counts.
# If all four are padded to the longest sequence:
# - what is the mask shape?
# - what fraction of (N,T) positions are padding?
#
# ES:
# Convierte [30 s, 45 s, 2 min, 4 h] a 30 fps en cantidades de fotogramas.
# Si las cuatro secuencias se rellenan hasta la más larga:
# - ¿cuál es la forma de la máscara?
# - ¿qué fracción de posiciones (N,T) corresponde a padding?
#
# TODO 6 / TAREA 6
#
# EN:
# Compare padding with keeping exactly 64 positions from every clip.
# What is gained? What temporal detail may be missed?
#
# ES:
# Compara padding con conservar exactamente 64 posiciones de cada clip.
# ¿Qué se gana? ¿Qué detalle temporal puede perderse?
#
# TODO 7 / TAREA 7
#
# EN:
# A surgical system has 3 synchronized cameras.
# Write:
# - one shape with camera as its own CAM axis;
# - one shape that folds camera into the batch axis.
# When must CAM remain explicit?
#
# ES:
# Un sistema quirúrgico tiene 3 cámaras sincronizadas.
# Escribe:
# - una forma con cámara como eje CAM independiente;
# - una forma donde CAM se integra en el eje de lote.
# ¿Cuándo debe mantenerse CAM explícito?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

durations_s = np.array([
    30,
    45,
    2 * 60,
    4 * 60 * 60,
])

fps = 30
lengths = durations_s * fps
T_max = int(lengths.max())

mask = np.zeros(
    (len(lengths), T_max),
    dtype=bool,
)

for i, n_frames in enumerate(lengths):
    mask[i, :int(n_frames)] = True

valid_fraction = mask.sum() / mask.size
padding_fraction = 1 - valid_fraction

print("Frame counts / Cantidad de fotogramas:", lengths.tolist())
print("Mask shape / Forma de máscara:", mask.shape)
print(f"Measured positions / Posiciones medidas: {valid_fraction:.2%}")
print(f"Padding positions / Posiciones de padding: {padding_fraction:.2%}")
print()

fig, ax = plt.subplots(figsize=(10, 2.8))

ax.imshow(
    mask,
    aspect="auto",
    cmap="Greys",
    interpolation="nearest",
)

ax.set_yticks(range(4))
ax.set_yticklabels([
    "30 s",
    "45 s",
    "2 min",
    "4 h",
])

ax.set_xlabel("Frame position T / Posición temporal T")
ax.set_title(
    f"Valid vs padding / Válido vs padding — "
    f"{padding_fraction:.2%} padding"
)

plt.tight_layout()
plt.show()

sampled_batch_shape = (4, 64, 224, 224, 3)

print("Fixed-sampling batch / Lote con muestreo fijo:", sampled_batch_shape)
print("EN: memory becomes bounded and predictable.")
print("ES: la memoria se vuelve acotada y predecible.")
print("EN: a long clip is represented by much more widely spaced observations.")
print("ES: un clip largo queda representado por observaciones mucho más separadas.")
print()

explicit_camera = (4, 3, 64, 224, 224, 3)  # N, CAM, T, H, W, C
folded_camera = (12, 64, 224, 224, 3)      # N*CAM, T, H, W, C

print("Camera explicit / Cámara explícita:", explicit_camera)
print("Axes / Ejes: (N, CAM, T, H, W, C)")
print()
print("Camera folded / Cámara integrada:", folded_camera)
print("Axes / Ejes: (N*CAM, T, H, W, C)")
print()
print("EN: keep CAM explicit when the model must know which synchronized view a frame came from or combine views structurally.")
print("ES: conserva CAM explícito cuando el modelo debe saber de qué vista sincronizada proviene un fotograma o combinar las vistas de forma estructurada.")

In [ ]:
# Exercise 3 design scenario, recomputed here in a visible cell so the three
# explorers below run whether or not the folded solution was executed. The
# valid-vs-padding heatmap, the printed step-by-step percentages, and the
# pad-vs-sample / camera-axis reasoning stay folded in the solution above.
durations_s = np.array([30, 45, 2 * 60, 4 * 60 * 60])
fps = 30
lengths = durations_s * fps
T_max = int(lengths.max())
padding_fraction = 1 - lengths.sum() / (len(lengths) * T_max)

sampled_batch_shape = (4, 64, 224, 224, 3)   # N, T, H, W, C
explicit_camera = (4, 3, 64, 224, 224, 3)    # N, CAM, T, H, W, C
folded_camera = (12, 64, 224, 224, 3)        # N*CAM, T, H, W, C

### Interactive duration and padding explorer / Explorador interactivo de duración y padding

Choose one of the four clip durations.

The notebook will show:

- its number of recorded frame positions at 30 fps;
- how much padding it would receive if the batch were padded to 4 hours;
- how sparse a 64-position sampling plan would be.

> 🇪🇸 Elige una de las cuatro duraciones.
>
> El cuaderno mostrará:
>
> - cuántas posiciones de fotogramas tendría a 30 fps;
> - cuánto padding recibiría si el lote se rellenara hasta 4 horas;
> - qué tan disperso sería un plan de muestreo de 64 posiciones.

In [ ]:
duration_selector = widgets.Dropdown(
    options=[
        ("30 seconds / 30 segundos", 0),
        ("45 seconds / 45 segundos", 1),
        ("2 minutes / 2 minutos", 2),
        ("4 hours / 4 horas", 3),
    ],
    value=0,
    description="Clip:",
    style={"description_width": "80px"},
)

def explore_duration(index):
    frames = int(lengths[index])
    padding = T_max - frames
    padding_for_clip = padding / T_max
    spacing = frames / 64

    print("Measured frame positions / Posiciones medidas:", frames)
    print("Padded to / Rellenado hasta:", T_max)
    print("Padding positions / Posiciones de padding:", padding)
    print(f"Padding fraction for this clip / Fracción de padding: {padding_for_clip:.2%}")
    print()

    print("If 64 positions are sampled / Si se muestrean 64 posiciones:")
    print(f"Average spacing / Separación media aproximada: {spacing:.1f} recorded frames")
    print()
    print("EN: padding preserves all measured positions but can waste representation space.")
    print("ES: el padding conserva todas las posiciones medidas, pero puede desperdiciar espacio de representación.")
    print("EN: fixed sampling bounds memory but observes long videos more sparsely.")
    print("ES: el muestreo fijo limita la memoria, pero observa los videos largos de forma más dispersa.")

duration_output = widgets.interactive_output(
    explore_duration,
    {"index": duration_selector},
)

display(widgets.VBox([duration_selector, duration_output]))

### Pad or sample? / ¿Padding o muestreo?

Switch between the two strategies.

There is no universally correct answer. The right choice depends on whether preserving every measured timestep is more important than bounding memory and computation.

> 🇪🇸 Cambia entre las dos estrategias.
>
> No existe una respuesta universalmente correcta. La decisión depende de si es más importante conservar todos los instantes medidos o limitar memoria y cómputo.

In [ ]:
strategy_toggle = widgets.ToggleButtons(
    options=[
        ("Padding", "padding"),
        ("Fixed 64 samples / 64 muestras fijas", "sampling"),
    ],
    value="padding",
    description="Strategy / Estrategia:",
    style={"description_width": "135px"},
)

def explain_strategy(strategy):
    if strategy == "padding":
        print("Shape concept / Concepto de forma: (N, T_max, H, W, C)")
        print("Mask / Máscara: (N, T_max)")
        print(f"Stress-test padding / Padding del escenario: {padding_fraction:.2%}")
        print("EN: measured timesteps are preserved, but padded slots are not observations.")
        print("ES: se conservan los instantes medidos, pero las posiciones de padding no son observaciones.")
    else:
        print("Example batch / Lote de ejemplo:", sampled_batch_shape)
        print("EN: every clip contributes exactly 64 sampled positions.")
        print("ES: cada clip aporta exactamente 64 posiciones muestreadas.")
        print("EN: memory is predictable, but events between sampled positions may be missed.")
        print("ES: la memoria es predecible, pero pueden perderse eventos entre posiciones muestreadas.")

strategy_output = widgets.interactive_output(
    explain_strategy,
    {"strategy": strategy_toggle},
)

display(widgets.VBox([strategy_toggle, strategy_output]))

### Camera-axis explorer / Explorador del eje de cámaras

A synchronized multi-camera system introduces a new design choice:

`(N, CAM, T, H, W, C)`

or:

`(N×CAM, T, H, W, C)`

Choose a representation below.

> 🇪🇸 Un sistema con varias cámaras sincronizadas introduce una nueva decisión:
>
> `(N,CAM,T,H,W,C)` o `(N×CAM,T,H,W,C)`.
>
> Elige una representación.

In [ ]:
camera_toggle = widgets.ToggleButtons(
    options=[
        ("Keep CAM explicit / Mantener CAM explícito", "explicit"),
        ("Fold CAM into batch / Integrar CAM al lote", "folded"),
    ],
    value="explicit",
    description="Camera / Cámara:",
    style={"description_width": "120px"},
)

def explain_camera(choice):
    if choice == "explicit":
        print("Shape / Forma:", explicit_camera)
        print("Axes / Ejes: (N, CAM, T, H, W, C)")
        print("EN: camera identity remains a separate coordinate.")
        print("ES: la identidad de la cámara permanece como una coordenada independiente.")
        print("EN: useful when synchronized views must be combined or compared.")
        print("ES: útil cuando las vistas sincronizadas deben combinarse o compararse.")
    else:
        print("Shape / Forma:", folded_camera)
        print("Axes / Ejes: (N*CAM, T, H, W, C)")
        print("EN: camera identity is no longer represented by a dedicated axis.")
        print("ES: la identidad de la cámara ya no está representada por un eje dedicado.")
        print("EN: this may be acceptable only if treating views as separate examples matches the task.")
        print("ES: puede ser adecuado solo si tratar las vistas como ejemplos separados coincide con la tarea.")

camera_output = widgets.interactive_output(
    explain_camera,
    {"choice": camera_toggle},
)

display(widgets.VBox([camera_toggle, camera_output]))

## What just happened / Qué acaba de pasar

You followed one **real verified video** from a file to a tensor and then used that concrete example to reason about larger systems.

### Five ideas to remember / Cinco ideas para recordar

1. **A video file is not yet a tensor.**  
   It must be decoded into frames and axes.

2. **Sampling changes what is observed.**  
   This pipeline retained 16 of 720 recorded frames — about **2.2%** of the recorded timesteps.

3. **The output shape depends on the question.**  
   A whole-video representation can summarize time; timestep labelling must preserve it.

4. **Padding and sampling solve different problems.**  
   Padding preserves measured positions but can waste space; fixed sampling controls memory but observes long sequences more sparsely.

5. **Camera, batch, and time are different semantic axes.**  
   They are not interchangeable just because they are dimensions.

### One-sentence takeaway / Idea en una frase

> **A good video pipeline makes every axis and every information-loss decision explicit.**

> 🇪🇸
>
> **Un buen pipeline de video hace explícito el significado de cada eje y cada decisión que elimina información.**


---

## Done with this section / Fin de esta sección

Next / Siguiente: **06 · Contraction with einsum / Contracción con einsum** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)